## Building Nonsequential Models Using Custom Modules

In [1]:
import torch
import torch.nn as nn

In [2]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [3]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = fetch_california_housing()
df = data.data
target = data.target

X_train, X_test, y_train, y_test = train_test_split(df, target, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_valid = torch.tensor(X_valid, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_valid = torch.tensor(y_valid, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=(device == "cuda"))

In [4]:
class WideAndDeep(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.deep_stack = nn.Sequential(
            nn.Linear(n_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU()
        )
        self.output_layer = nn.Linear(40 + n_features, 1)
    
    def forward(self, X):
        deep_output = self.deep_stack(X)
        wide_and_deep = torch.concat([X, deep_output], dim=1)
        return self.output_layer(wide_and_deep)

In [5]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        mean_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [6]:
torch.manual_seed(42)
n_features = X_train.shape[1]
learning_rate = 0.002

model = WideAndDeep(n_features).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
n_epochs = 20

In [7]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 1.3319
Epoch 2/20, Loss: 0.6117
Epoch 3/20, Loss: 0.5662
Epoch 4/20, Loss: 0.5345
Epoch 5/20, Loss: 0.5104
Epoch 6/20, Loss: 0.4929
Epoch 7/20, Loss: 0.4804
Epoch 8/20, Loss: 0.4683
Epoch 9/20, Loss: 0.4589
Epoch 10/20, Loss: 0.4501
Epoch 11/20, Loss: 0.4423
Epoch 12/20, Loss: 0.4354
Epoch 13/20, Loss: 0.4282
Epoch 14/20, Loss: 0.4228
Epoch 15/20, Loss: 0.4167
Epoch 16/20, Loss: 0.4116
Epoch 17/20, Loss: 0.4075
Epoch 18/20, Loss: 0.4018
Epoch 19/20, Loss: 0.3971
Epoch 20/20, Loss: 0.3934


In [8]:
X_test = X_test.to(device)
y_test = y_test.to(device)

with torch.no_grad():
    y_test_pred = model(X_test)

mse(y_test, y_test_pred)

tensor(0.4076, device='cuda:0')

What if you want to send a subset of the features through the wide path and a different subset through the deep path, best approach is to let the model take two separate tensors.

### Building Models with Multiple Inputs